# Notebook 03 — Extracting vibes with Claude

This notebook enriches the ~40 selected Lyon restaurants with LLM-generated
content that's missing from the raw TripAdvisor data:

- **vibes**: 3-5 atmosphere tags (e.g. "cozy", "romantic", "industrial-chic")
- **description**: a 2-3 sentence evocative summary
- **signatures**: 1-3 likely signature dishes
- **shortcomings**: 1-2 honest critiques (for trust)

We call `claude-haiku-4-5` once per restaurant. Results are saved
incrementally to `data/processed/lyon_restaurants_with_vibes.json` so a
crash mid-batch doesn't lose progress.

**Estimated cost**: ~0.30 USD for 40 restaurants.

In [1]:
"""Notebook 03 — Extract vibes from Lyon restaurants using Claude Haiku."""

import json
import os
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from tqdm import tqdm

# Load API key from .env at project root
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

# Paths
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INPUT_CSV = DATA_PROCESSED / "lyon_restaurants_selected.csv"
OUTPUT_JSON = DATA_PROCESSED / "lyon_restaurants_with_vibes.json"

# Anthropic client
client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 1024

# Sanity check
print(f"API key loaded: {bool(os.environ.get('ANTHROPIC_API_KEY'))}")
print(f"Input file: {INPUT_CSV}")
print(f"Output file: {OUTPUT_JSON}")

API key loaded: True
Input file: C:\Users\busar\Desktop\connoisseur\data\processed\lyon_restaurants_selected.csv
Output file: C:\Users\busar\Desktop\connoisseur\data\processed\lyon_restaurants_with_vibes.json


In [2]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} restaurants")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded 148 restaurants
Columns: ['restaurant_name', 'address', 'primary_cuisine', 'cuisines', 'price_level', 'avg_rating', 'total_reviews_count', 'top_tags', 'awards', 'special_diets', 'features', 'open_days_per_week']


,restaurant_name,address,primary_cuisine,cuisines,price_level,avg_rating,total_reviews_count,top_tags,awards,special_diets,features,open_days_per_week
0,Lyon-Dakar,"227 rue de Crequi, 69003 Lyon France",African,African,€€-€€€,4.0,189.0,"Mid-range, African","Certificate of Excellence 2019, Certificate of...",NaN,NaN,5.0
1,Mattsam Restaurant Messob,"85 rue Massena, 69006 Lyon France",African,"African, Ethiopian",€€-€€€,4.0,140.0,"Mid-range, African, Ethiopian, Vegetarian Frie...","Travellers' Choice, Certificate of Excellence ...","Vegetarian Friendly, Vegan Options",NaN,NaN
2,La Mangue Amère,"7 rue du Jardin des Plantes, 69001 Lyon France",African,African,€€-€€€,4.0,91.0,"Mid-range, African","Travellers' Choice, Certificate of Excellence ...",NaN,NaN,7.0


In [3]:
SYSTEM_PROMPT = """You are a perceptive restaurant critic with deep knowledge of \
European dining culture, especially French gastronomy and Lyon's "bouchon" tradition.

Your task: given factual data about a Lyon restaurant (name, cuisine, price level, \
rating, address), generate ENRICHED descriptive content. You're not making things up \
wildly — you're inferring plausible details from the factual signals.

CRITICAL RULES:
- Output STRICTLY valid JSON, nothing else (no markdown, no preamble).
- Write in English, except for proper nouns and dish names that should stay in French.
- Be evocative but grounded — no clichés like "hidden gem" or "must-try".
- The "shortcomings" field is essential for honesty — every restaurant has flaws."""


def build_user_prompt(row: pd.Series) -> str:
    """Build the user prompt from one restaurant row."""
    return f"""Restaurant data:
- Name: {row.get('restaurant_name', 'Unknown')}
- Primary cuisine: {row.get('primary_cuisine', 'Unknown')}
- All cuisines: {row.get('cuisines', '')}
- Price level: {row.get('price_level', 'Unknown')}
- Rating: {row.get('avg_rating', '?')}/5
- Number of reviews: {row.get('total_reviews_count', '?')}
- Address: {row.get('address', '')}

Generate the enriched profile as JSON with exactly these keys:
{{
  "vibes": [list of 3 to 5 atmosphere tags, single lowercase words or short phrases],
  "description": "a 2-3 sentence evocative summary of the place and what makes it distinctive",
  "signatures": [list of 1 to 3 likely signature dishes, in French when appropriate],
  "shortcomings": [list of 1 to 2 honest critiques, e.g. service quirks, price/value, crowds]
}}

Examples of good vibes: "moody", "industrial-chic", "old-world charm", "buzzing", \
"intimate", "terrace-perfect", "family-friendly", "trendy".

Examples of bad vibes: "amazing", "the best", "incredible" (too vague and promotional).

Output ONLY the JSON, nothing else."""

In [7]:
# Test the prompt on the first restaurant
test_row = df.iloc[0]
test_prompt = build_user_prompt(test_row)

print("=" * 70)
print("USER PROMPT:")
print("=" * 70)
print(test_prompt)
print("\n" + "=" * 70)
print("CLAUDE RESPONSE:")
print("=" * 70)

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": test_prompt}],
)

# Extract text
response_text = response.content[0].text
print(response_text)

# Try to parse as JSON, with defensive stripping
print("\n" + "=" * 70)
print("PARSED:")
print("=" * 70)

text = response_text.strip()

# Strip markdown code fences if present
if text.startswith("```"):
    # Remove first line (```json or ```)
    text = text.split("\n", 1)[1] if "\n" in text else text
    # Remove trailing ```
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()

parsed = json.loads(text)
print(json.dumps(parsed, indent=2))

# Print usage stats
print(f"\nTokens used: input={response.usage.input_tokens}, output={response.usage.output_tokens}")

USER PROMPT:
Restaurant data:
- Name: Lyon-Dakar
- Primary cuisine: African
- All cuisines: African
- Price level: €€-€€€
- Rating: 4.0/5
- Number of reviews: 189.0
- Address: 227 rue de Crequi, 69003 Lyon France

Generate the enriched profile as JSON with exactly these keys:
{
  "vibes": [list of 3 to 5 atmosphere tags, single lowercase words or short phrases],
  "description": "a 2-3 sentence evocative summary of the place and what makes it distinctive",
  "signatures": [list of 1 to 3 likely signature dishes, in French when appropriate],
  "shortcomings": [list of 1 to 2 honest critiques, e.g. service quirks, price/value, crowds]
}

Examples of good vibes: "moody", "industrial-chic", "old-world charm", "buzzing", "intimate", "terrace-perfect", "family-friendly", "trendy".

Examples of bad vibes: "amazing", "the best", "incredible" (too vague and promotional).

Output ONLY the JSON, nothing else.

CLAUDE RESPONSE:
```json
{
  "vibes": ["convivial", "unpretentious", "spirited", "diver

In [8]:
def enrich_one_restaurant(row: pd.Series, max_retries: int = 2) -> dict | None:
    """
    Call Claude to enrich one restaurant. Returns the parsed JSON dict, or None on failure.
    Retries up to max_retries on JSON parse errors or API hiccups.
    """
    user_prompt = build_user_prompt(row)
    
    for attempt in range(max_retries + 1):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
            )
            text = response.content[0].text.strip()
            
            # Sometimes Claude wraps JSON in ```json ... ``` despite instructions
            if text.startswith("```"):
                text = text.split("```")[1]
                if text.startswith("json"):
                    text = text[4:]
                text = text.strip()
            
            return json.loads(text)
        
        except json.JSONDecodeError as e:
            if attempt < max_retries:
                print(f"  ⚠️ JSON parse failed (attempt {attempt+1}), retrying...")
                continue
            print(f"  ❌ Final JSON parse failure: {e}")
            print(f"     Raw response: {text[:200]}")
            return None
        
        except Exception as e:
            print(f"  ❌ API error: {e}")
            return None
    
    return None

In [9]:
# Load any previously-saved results so we can resume from where we stopped
if OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        results = json.load(f)
    done_ids = {r["restaurant_link"] for r in results if "restaurant_link" in r}
    print(f"Resuming: {len(results)} restaurants already enriched")
else:
    results = []
    done_ids = set()
    print("Starting from scratch")

Starting from scratch


In [10]:
# Pick the column used as unique ID
ID_COL = "restaurant_link"  # adjust if your dataset uses something else

# Iterate with a progress bar
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Enriching"):
    restaurant_id = row.get(ID_COL, str(idx))
    
    # Skip if already done
    if restaurant_id in done_ids:
        continue
    
    enrichment = enrich_one_restaurant(row)
    if enrichment is None:
        print(f"  Skipping {row['restaurant_name']} due to errors")
        continue
    
    # Build the full record: original data + enrichment
    record = {
        "restaurant_link": restaurant_id,
        "restaurant_name": row.get("restaurant_name"),
        "primary_cuisine": row.get("primary_cuisine"),
        "cuisines": row.get("cuisines"),
        "price_level": row.get("price_level"),
        "avg_rating": row.get("avg_rating"),
        "total_reviews_count": row.get("total_reviews_count"),
        "address": row.get("address"),
        # Enrichment fields from Claude
        "vibes": enrichment.get("vibes", []),
        "description": enrichment.get("description", ""),
        "signatures": enrichment.get("signatures", []),
        "shortcomings": enrichment.get("shortcomings", []),
    }
    results.append(record)
    
    # Save after every successful enrichment (so a crash doesn't lose progress)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✅ Done. {len(results)} restaurants enriched.")

Enriching: 100%|████████████████████████████████████████████████████████████████████████████████████| 148/148 [08:42<00:00,  3.53s/it]


✅ Done. 148 restaurants enriched.


In [11]:
import random

# Pick 3 random restaurants to inspect
sample = random.sample(results, min(3, len(results)))

for r in sample:
    print(f"\n{'='*70}")
    print(f"🍽  {r['restaurant_name']}")
    print(f"   {r['primary_cuisine']} · {r['price_level']} · {r['avg_rating']}/5")
    print(f"   {r['address']}")
    print(f"\n📝 {r['description']}")
    print(f"\n🎭 Vibes: {', '.join(r['vibes'])}")
    print(f"\n🍴 Signatures: {', '.join(r['signatures'])}")
    print(f"\n⚠️  Shortcomings: {' | '.join(r['shortcomings'])}")


🍽  L'Etoile d'Asie
   Asian · €€-€€€ · 4.5/5
   13 rue Cavenne, 69007 Lyon France

📝 L'Etoile d'Asie brings Vietnamese and broader Asian cooking to the Confluence district with a refreshingly straightforward approach—no fusion theater, just solid technique and genuine flavours. The 4.5-star rating across 288 reviews suggests consistency and word-of-mouth credibility in a neighbourhood increasingly curious about authentic regional Asian kitchens.

🎭 Vibes: casual-intimate, neighbourhood-focused, buzzing, unpretentious, warm

🍴 Signatures: Phở bœuf, Rouleaux de printemps, Crevettes sautées aux germes de soja

⚠️  Shortcomings: Service can lag during peak dinner hours, reflecting modest staffing typical of mid-range neighbourhood spots | Limited wine pairing options—the €€-€€€ bracket doesn't accommodate the wine-forward expectations some Lyon diners bring to the table

🍽  Hank Burger Lyon Opera
   Street Food · €€-€€€ · 4.5/5
   5 rue Pizay, 69001 Lyon France

📝 Hank Burger Lyon Opera b

In [12]:
# Some basic statistics on the enrichment
total = len(results)
all_vibes = [v for r in results for v in r["vibes"]]
unique_vibes = set(all_vibes)

print(f"Total enriched: {total}")
print(f"Total vibe tags: {len(all_vibes)} ({len(unique_vibes)} unique)")
print(f"Average vibes per restaurant: {len(all_vibes) / total:.1f}")

# Top vibes
vibe_counts = pd.Series(all_vibes).value_counts()
print(f"\nTop 15 vibes across all restaurants:")
print(vibe_counts.head(15))

Total enriched: 148
Total vibe tags: 635 (209 unique)
Average vibes per restaurant: 4.3

Top 15 vibes across all restaurants:
convivial               68
unpretentious           55
intimate                37
casual                  26
neighborhood-focused    23
casual-elegant          18
buzzing                 17
contemporary            16
bustling                13
lively                  12
approachable            11
cosmopolitan            10
vibrant                  9
aromatic                 8
wine-focused             7
Name: count, dtype: int64
